# NB-06 — LLM Backbone Integration

**Goal:** Connect CLIP + projector + Qwen2-VL and run your first multimodal QA.

Two paths:
1. **Native** — Qwen2-VL built-in vision (baseline)
2. **Custom** — our CLIP + MLP projector → `inputs_embeds` (learning pipeline)

---

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import time
import torch
import requests
from io import BytesIO
from PIL import Image
from dotenv import load_dotenv

load_dotenv(Path("..") / ".env")

from src.llm.backbone import MultimodalLLM

DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

In [ ]:
def load_image(url: str) -> Image.Image:
  resp = requests.get(url, timeout=15)
  return Image.open(BytesIO(resp.content)).convert("RGB")

IMG_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/320px-Cute_dog.jpg"
image = load_image(IMG_URL)
question = "What animal is in this image? Answer briefly."
image.resize((224, 224))

## 1. Load pipeline from config

First run downloads ~4GB (CLIP + Qwen2-VL-2B). Use `encoder_on_cpu=True` if GPU memory is tight.

In [ ]:
CONFIG = Path("..") / "config" / "model_config.yaml"

print("Loading MultimodalLLM (this may take a few minutes)...")
model = MultimodalLLM.from_config(
    CONFIG,
    device=DEVICE,
    encoder_on_cpu=(DEVICE == "mps"),  # save MPS memory for LLM
)
print(f"LLM hidden size: {model.llm_hidden_size}")

## 2. Native baseline (Qwen2-VL vision tower)

In [ ]:
t0 = time.perf_counter()
native_answer = model.generate_native(image, question, max_new_tokens=128)
native_latency = time.perf_counter() - t0

print(f"Latency: {native_latency:.2f}s")
print(f"Answer (native): {native_answer}")

## 3. Custom pipeline (CLIP + MLP projector)

In [ ]:
inputs = model.prepare_inputs(image, question)
print("Merged input layout: [visual_tokens] + [text_tokens]")
print(f"  visual tokens: {inputs['num_visual_tokens']}")
print(f"  text tokens:   {inputs['num_text_tokens']}")
print(f"  total tokens:  {inputs['total_tokens']}")
print(f"  inputs_embeds: {tuple(inputs['inputs_embeds'].shape)}")

In [ ]:
t0 = time.perf_counter()
custom_answer = model.generate(image, question, max_new_tokens=128)
custom_latency = time.perf_counter() - t0

print(f"Latency: {custom_latency:.2f}s")
print(f"Answer (custom): {custom_answer}")

## 4. Side-by-side comparison

In [ ]:
print("=" * 60)
print(f"Question: {question}")
print("-" * 60)
print(f"Native ({native_latency:.1f}s): {native_answer}")
print(f"Custom ({custom_latency:.1f}s): {custom_answer}")
print("=" * 60)
print("\nNote: Custom path uses untrained CLIP+projector — quality improves after fine-tuning (Phase 5).")

## 5. Multi-turn chat

In [ ]:
history = [
    {"role": "user", "content": "What is in this image?"},
]
reply1 = model.chat(history, image=image, max_new_tokens=80, use_native=True)
print("Turn 1:", reply1)

history.append({"role": "assistant", "content": reply1})
history.append({"role": "user", "content": "What color is it mainly?"})
reply2 = model.chat(history, image=image, max_new_tokens=80, use_native=True)
print("Turn 2:", reply2)

## 6. Edge cases

In [ ]:
# Text-only (no image)
text_only = model.generate(None, "What is the capital of France?", max_new_tokens=32)
print("Text-only:", text_only)

# Large image (auto-resized by CLIP processor)
large = image.resize((512, 512))
large_inputs = model.prepare_inputs(large, question)
print(f"Large image visual tokens: {large_inputs['num_visual_tokens']}")

## 7. Memory benchmark (optional)

In [ ]:
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    _ = model.generate_native(image, question, max_new_tokens=64)
    peak_mb = torch.cuda.max_memory_allocated() / 1e6
    print(f"Peak GPU memory: {peak_mb:.0f} MB")
elif torch.backends.mps.is_available():
    print("MPS memory stats not available — monitor Activity Monitor during runs.")
else:
    print("Running on CPU — expect slower inference.")

## ✅ Phase 3 Checklist

- [ ] Native Qwen2-VL baseline produces a sensible answer
- [ ] Custom path shows merged `inputs_embeds` shapes
- [ ] Understand visual tokens prepended before text tokens
- [ ] Multi-turn chat works with `model.chat()`

**Next:** Phase 4 — `MultimodalQAPipeline` in `src/pipeline/multimodal_qa.py`